# Template for Simple Customer Service Agent
_Build a simple customer service agent using LangChain Framework._

This minimal customer service agent should support only the following service requests from its customers:
1. Cancel order
2. Update delivery address

Each request should be handled independently, and required parameters should always be collected fresh for that request.

In [ ]:
# First import function "tool" from module langchain_core.tools, 
# class "SystemMessage" and "HumanMessage" from module "langchain_core.messages" and 
# class "ChatOllama" from module "langchain_ollama.chat_models"

# Imports packages

from langchain_core.tools import tool
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_ollama.chat_models import ChatOllama

In [ ]:
# Set variables over the following information.
# Ollama endpoint is available at http://localhost:11434.
# Use model Llama 3.2 3B over name "llama3.2:3b"

# Sets endpoints for Ollama models to be available over web requests.
 
OLLAMA_MODEL = "llama3.2:3b"
OLLAMA_ENDPOINT = "http://localhost:11434/"

## Tools

Use decorator to convert Python functions to LangChain tools that the agent will be using.

These tools should be dummy for this lab. experiment. But these will have real implementation for production systems.

In [ ]:
# Write two functions - one for order cancellation and other one is for address change

# Order Cancellation Function: It should have an input parameter for order id of type string
# and the function should return a message indicating successful cancellation of a particular order.
# The function should have an appropriate description for model to understand
# The function should be decorated using "@tool" to make it LangChain compatible.

@tool
def cancel_order(order_id: str) -> str:
    """Cancels an existing order using order_id."""
    return f"[SUCCESS] Order {order_id} has been cancelled."

@tool
def update_delivery_address(order_id: str, new_delivery_address: str) -> str:
    """Updates delivery address for an existing order."""
    return f"[SUCCESS] Delivery address was updated for order {order_id}. New address is {new_delivery_address}"

tools = {
    "cancel_order": cancel_order,
    "update_delivery_address": update_delivery_address
}

## Chat Model

In [7]:
# Instruction to be used to set agent's behaviour

agent_instructions = """You are a customer service agent for an eCommerce platform.
You can only help with three services:
1) cancel order
2) update delivery address
3) issue refund

Rules:
- Keep responses brief and clear.
- If a request is outside these three services, politely decline and restate supported services.
- Never invent order_id or new_delivery_address.
- Do not call tools until all required fields are explicitly provided by the customer.
- For every new request, fetch parameter values from the customer again. Do not reuse values from prior turns.

For cancel order:
- Required field: order_id
- If order_id is missing, ask only for order_id.
- If order_id is present, call cancel_order(order_id) directly.

For update delivery address:
- Required fields: order_id and new_delivery_address
- If any is missing, ask follow-up for missing fields only.
- If both are present, call update_delivery_address(order_id, new_delivery_address) directly.

For issue refund:
- Required fields: order_id and amount
- If the amount is not available, default to $100.
- Call issue_refund(order_id, amount) directly.

After tool execution:
- Show the tool confirmation message to the customer.
"""

In [ ]:
# Instantiate a chat client by calling constructor of class "ChatOllama" passing
# model name as argument to parameter "model",
# argument "False" to argument "reasoning", and
# Ollama endpoint as argument to parameter "base_url"

# Initializes chat client
client = ChatOllama(model=OLLAMA_MODEL, 
                    reasoning=False,
                    base_url=OLLAMA_ENDPOINT
                    )

# Binds tools to the chat client
client_with_tools = client.bind_tools([cancel_order, update_delivery_address])

In [ ]:
# Maintain a list of all messages that flow across interactions
# Initialize the message list with first message of type SystemMessage.
# The constructor of the system message should be called with agent instructions
# as an argument to the parameter "content".

# Maintains a list of all messages that flow across interactions

messages = [
    SystemMessage(content=agent_instructions)   # Initializes with intruction (as a SystemMessage) that is specific to agent.
    ]

In [ ]:
# Check if the message list initization is correct by displaying the list by 
# calling function "display" and passing message list as its first parameter

display(messages)        # Checks structure of first message i.e. SystemMessage

In [ ]:
query = "I want to cancel my order 'Order_12345'."      # Customer query to be sent to client


# Consider the above customer query.
# Initialize a message of type "HumanMessage" passing the above
# query to the constructor of class and
# append that message instance to the message list.

messages.append(HumanMessage(query))                    # Appends customer query (as a HumanMessage) into message list

In [ ]:
# Check if the message list contains both the messages by dislaying it

display(messages)       # Checks structure of second message i.e. HumanMessage

In [ ]:
# Now, pass the message list to the tools-bound chat client by calling
# its function "invoke" passing the message list into it.
# Store the returned message in a variable.

# Passes messages (as a single input) to model and receives response
client_message = client_with_tools.invoke(messages)

In [ ]:
# Display the model-returned message to analyze element `tool_calls` to check
# if model has returned the correct tool name(s) with argument(s) for client to call those later

# Displays the (structured) model-returned message to analyze element `tool_calls` to check
# if model has returned the correct tool name(s) with argument(s) for client to call those later
display(client_message)

In [ ]:
# Append last received message into message list

messages.append(client_message)         # Appends last received message into message list

In [ ]:
# Now, loop over all the tool calls. It is a list that can be accessed over 
# property "tool_calls" of the last client returned message.

# For each tool call (which will be a dictionary), do the followings.
#   Get name of the tool from its "name" property
#   Get the tool (function pointer) from the (already prepared) mapping
#   Invoke the tool by calling function "invoke" on the tool object 
#       passing tool call dictionary and store the tool output as tool message
#   Append the tool message into message list

for tool_call in client_message.tool_calls:
    tool_name = tool_call["name"].lower()               # Extracts tool name and converts to lower case
    selected_tool = tools[tool_name]                    # Gets pointer to tool
    tool_message = selected_tool.invoke(tool_call)      # Invokes (or calls) the tool with arguments

    # Prints tool name, its arguments and its output for analysis
    print(f"Tool: '{tool_call["name"]}', \nArguments: {tool_call["args"]}, \nTool Output: {tool_message.content}")

    messages.append(tool_message)                       # Message received from tool gets added to message list

In [ ]:
# To get the final message from the model, call function "invoke" of the tools-bound client
# passing message list into it and store the return message and print it as final message to the customer

final_response = client_with_tools.invoke(messages)         # Passes all messages back to model to generate final response
print(f"Final Agent Response: {final_response.content}")    # Prints final response that the customer would see

messages.append(final_response)      # Appends final response into message list
display(messages)                    # Displays the final message list for analysis